# 01 - Preprocessing

### Austin Melendez & Sara Bruggman
###### Last Updated: 4/28/2026 8:44am

This notebook applies the preserved scoring and classification rules, and writes reusable cleaned datasets for later notebooks.


In [ ]:
suppressPackageStartupMessages({
  library(tidyverse)
  library(stringr)
  library(forcats)
})

## Inputs and Outputs

Input: `shelby_anon_data.csv`

Outputs written to `data/processed/`:

* `survey_clean.csv`: long-format cleaned data, one row per participant per survey time
* `pre_survey_clean.csv`: cleaned Pre rows
* `post_survey_clean.csv`: cleaned Post rows
* `matched_change.csv`: one row per matched participant with Pre, Post, and change scores
* `analysis_data.rds`: an R list containing all cleaned dataframes and preprocessing metadata


In [ ]:
raw_path <- "./shelby_anon_data.csv"
processed_dir <- file.path("./data/")
dir.create(processed_dir, recursive = TRUE, showWarnings = FALSE)

raw_data <- read_csv(file.path(raw_path))

excluded_ids <- c(40, 43, 81, 102, 107, 125, 126, 144, 149)

clean_base <- raw_data %>%
  select(-matches("^\\.\\.\\.[0-9]+$"), -any_of("NOTES:")) %>%
  filter(!`DID #` %in% excluded_ids)


New names:
• `` -> `...51`
Rows: 462 Columns: 51
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (47): MAJOR, CONCENTRATION, Class Level, Gender, Race/Ethnicity, First G...
dbl  (3): DID #, Age at census, Cumulative GPA
lgl  (1): ...51

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


## Score Jefferson and Toronto Items

The Jefferson and Toronto scales are scored independently because they have different item counts and response ranges.


In [ ]:
J_items <- paste0("J", 1:14)
T_items <- paste0("T", 1:16)

J_reverse <- c("J1", "J3", "J6", "J7", "J8", "J11", "J13")
T_reverse <- c("T2", "T3", "T7", "T10", "T11", "T12", "T14", "T15")

likert_levels_7 <- c(
  "Extremely Disagree",
  "Disgaree",
  "Somewhat Disagree",
  "Neutral",
  "Somewhat Agree",
  "Agree",
  "Extremely Agree"
)

toronto_levels_5 <- c(
  "Disgaree",
  "Somewhat Disagree",
  "Neutral",
  "Somewhat Agree",
  "Agree"
)

J_survey <- clean_base %>%
  select(all_of(J_items)) %>%
  mutate(across(everything(), ~ as.numeric(factor(.x, levels = likert_levels_7, ordered = TRUE)))) %>%
  mutate(across(all_of(J_reverse), ~ 8 - .x))

T_survey <- clean_base %>%
  select(all_of(T_items)) %>%
  mutate(across(everything(), ~ as.numeric(factor(.x, levels = toronto_levels_5, ordered = TRUE)) - 1)) %>%
  mutate(across(all_of(T_reverse), ~ 4 - .x))

scored_data <- clean_base %>%
  mutate(
    J_total = rowSums(J_survey, na.rm = TRUE),
    T_total = rowSums(T_survey, na.rm = TRUE)
  )


## Standardize Majors, STEM Status, and Condition

The preserved project rule is: standardized `Biological Sciences` is `STEM`; all other standardized majors are `Non-STEM`.


In [ ]:
survey_clean <- scored_data %>%
  mutate(
    # Standardize Majors
    Major = case_when(
      str_detect(MAJOR, regex("health science", ignore_case = TRUE)) ~ "Health Sciences",
      str_detect(MAJOR, regex("nursing", ignore_case = TRUE)) ~ "Nursing",
      str_detect(MAJOR, regex("biological science", ignore_case = TRUE)) ~ "Biological Sciences",
      str_detect(MAJOR, regex("kinesiology", ignore_case = TRUE)) ~ "Kinesiology",
      str_detect(MAJOR, regex("public health", ignore_case = TRUE)) ~ "Public Health",
      str_detect(MAJOR, regex("nutrition", ignore_case = TRUE)) ~ "Nutrition",
      str_detect(MAJOR, regex("recreation therapy", ignore_case = TRUE)) ~ "Recreation Therapy",
      str_detect(MAJOR, regex("undeclared", ignore_case = TRUE)) ~ "Undeclared",
      TRUE ~ "Undeclared"
    ),
    # Classify STEM v Non-STEM
    STEM = if_else(Major %in% c("Biology", "Biological Sciences"), "STEM", "Non-STEM"),
    Condition = case_when(
      `Intervention v Control` == "I" ~ "Intervention",
      `Intervention v Control` == "C" ~ "Control",
      TRUE ~ NA
    ),
    # Convert variables to factors
    Survey = factor(Survey, levels = c("Pre", "Post")),
    STEM = factor(STEM, levels = c("STEM", "Non-STEM")),
    Condition = factor(Condition, levels = c("Control", "Intervention")),
    Gender = fct_collapse(as.factor(Gender),
      "Man" = "Man",
      "Woman" = "Woman",
      "Other" = c("Other", "Other/Unknown", "Another Gender")
    ),
    # Change N/As, unknown, or missing values to NA
    `Transfer Student` = na_if(`Transfer Student`, "N/A"),
    `Race/Ethnicity` = na_if(`Race/Ethnicity`, "Unknown"),
    `Class Level` = na_if(`Class Level`, "N/A")
  ) %>%
    # Select final variables
  select(
    `DID #`, Survey, Condition,
    `First Generation Status`, Gender, `Transfer Student`, `Upper v Lower`,
    `Race/Ethnicity`, `Class Level`, Major, STEM, J_total, T_total
  ) %>%
    # Convert other demographic variables to factors
  mutate(across(c(`First Generation Status`, `Transfer Student`, `Upper v Lower`, `Race/Ethnicity`, `Class Level`), as.factor))


Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `Gender = fct_collapse(...)`.
Caused by warning:
! Unknown levels in `f`: Other, Other/Unknown”


## Create Matched Pre/Post Data

Later notebooks use both long-format data and a one-row-per-participant change-score dataset. Matching is done by `DID #`.


In [ ]:
pre_survey_clean <- survey_clean %>%
  filter(Survey == "Pre") %>%
  arrange(`DID #`)

post_survey_clean <- survey_clean %>%
  filter(Survey == "Post") %>%
  arrange(`DID #`)

matched_ids <- intersect(pre_survey_clean$`DID #`, post_survey_clean$`DID #`)

pre_matched <- pre_survey_clean %>%
  filter(`DID #` %in% matched_ids) %>%
  arrange(`DID #`)

post_matched <- post_survey_clean %>%
  filter(`DID #` %in% matched_ids) %>%
  arrange(`DID #`)

stopifnot(identical(pre_matched$`DID #`, post_matched$`DID #`))

matched_change <- pre_matched %>%
  transmute(
    `DID #`, Condition, STEM, Major, Gender,
    `Transfer Student`, `Upper v Lower`, `Race/Ethnicity`, `Class Level`, `First Generation Status`,
    J_pre = J_total,
    J_post = post_matched$J_total,
    J_change = J_post - J_pre,
    T_pre = T_total,
    T_post = post_matched$T_total,
    T_change = T_post - T_pre
  )

participant_time_counts <- survey_clean %>%
  count(`DID #`, Survey) %>%
  tidyr::pivot_wider(names_from = Survey, values_from = n, values_fill = 0)

condition_consistency <- survey_clean %>%
  group_by(`DID #`) %>%
  summarise(n_conditions = n_distinct(Condition, na.rm = TRUE), .groups = "drop")


## Save Reusable Data

Future notebooks should load `data/analysis_data.rds` when working in R, or the individual CSV files when a portable text format is preferred.


In [ ]:
write.csv(survey_clean, file.path(processed_dir, "survey_clean.csv"), row.names = FALSE)
write.csv(pre_survey_clean, file.path(processed_dir, "pre_survey_clean.csv"), row.names = FALSE)
write.csv(post_survey_clean, file.path(processed_dir, "post_survey_clean.csv"), row.names = FALSE)
write.csv(matched_change, file.path(processed_dir, "matched_change.csv"), row.names = FALSE)

analysis_data <- list(
  survey_clean = survey_clean,
  pre_survey_clean = pre_survey_clean,
  post_survey_clean = post_survey_clean,
  matched_change = matched_change,
  metadata = list(
    excluded_ids = excluded_ids,
    J_reverse = J_reverse,
    T_reverse = T_reverse,
    participant_time_counts = participant_time_counts,
    condition_consistency = condition_consistency
  )
)

saveRDS(analysis_data, file.path(processed_dir, "analysis_data.rds"))


## Preprocessing Checks


In [ ]:
list(
  retained_rows = nrow(survey_clean),
  matched_participants = nrow(matched_change),
  survey_counts = table(survey_clean$Survey),
  stem_counts = table(matched_change$STEM),
  condition_counts = table(matched_change$Condition),
  condition_by_stem = table(matched_change$STEM, matched_change$Condition),
  non_one_to_one_pre_post = participant_time_counts %>% filter(Pre != 1 | Post != 1),
  inconsistent_condition_ids = condition_consistency %>% filter(n_conditions != 1)
)


$retained_rows
[1] 444

$matched_participants
[1] 222

$survey_counts

 Pre Post 
 222  222 

$stem_counts

    STEM Non-STEM 
     100      122 

$condition_counts

     Control Intervention 
         126           96 

$condition_by_stem
          
           Control Intervention
  STEM          47           53
  Non-STEM      79           43

$non_one_to_one_pre_post
# A tibble: 0 × 3
# ℹ 3 variables: DID # <dbl>, Pre <int>, Post <int>

$inconsistent_condition_ids
# A tibble: 0 × 2
# ℹ 2 variables: DID # <dbl>, n_conditions <int>


In [ ]:
table(matched_change$STEM, matched_change$Condition)

          
           Control Intervention
  STEM          47           53
  Non-STEM      79           43

In [ ]:
table(matched_change$Gender, matched_change$Condition)

       
        Control Intervention
  Other       1            0
  Man        23           22
  Woman     102           73

In [ ]:
table(matched_change$`Upper v Lower`, matched_change$Condition)

   
    Control Intervention
  L      58           52
  U      68           44

In [ ]:
table(matched_change$`Transfer Student`, matched_change$Condition)

     
      Control Intervention
  No       79           55
  Yes      47           40

In [ ]:
table(matched_change$`Race/Ethnicity`, matched_change$Condition)

                  
                   Control Intervention
  African American       5            4
  American Indian        1            0
  Asian                 38           28
  Hispanic/Latino       32           33
  Pacific Islander       3            1
  Two or More           18            7
  White                 28           20

In [ ]:
table(matched_change$`Class Level`, matched_change$Condition)

           
            Control Intervention
  Freshmen        4            1
  Junior         28           29
  Senior         61           52
  Sophomore      32           13

In [ ]:
table(matched_change$`First Generation Status`, matched_change$Condition)

     
      Control Intervention
  No       91           67
  Yes      35           28

In [ ]:
head(survey_clean)

DID #,Survey,Condition,First Generation Status,Gender,Transfer Student,Upper v Lower,Race/Ethnicity,Class Level,Major,STEM,J_total,T_total
<dbl>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<chr>,<fct>,<dbl>,<dbl>
1,Pre,Control,Yes,Man,No,L,Asian,Sophomore,Nursing,Non-STEM,87,47
1,Post,Control,Yes,Man,No,L,Asian,Sophomore,Nursing,Non-STEM,96,48
2,Pre,Control,Yes,Man,Yes,L,Hispanic/Latino,Senior,Public Health,Non-STEM,71,42
2,Post,Control,Yes,Man,Yes,L,Hispanic/Latino,Senior,Public Health,Non-STEM,79,45
3,Pre,Control,No,Woman,Yes,U,Asian,Junior,Biological Sciences,STEM,80,50
3,Post,Control,No,Woman,Yes,U,Asian,Junior,Biological Sciences,STEM,95,53


In [ ]:
head(pre_survey_clean)

DID #,Survey,Condition,First Generation Status,Gender,Transfer Student,Upper v Lower,Race/Ethnicity,Class Level,Major,STEM,J_total,T_total
<dbl>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<chr>,<fct>,<dbl>,<dbl>
1,Pre,Control,Yes,Man,No,L,Asian,Sophomore,Nursing,Non-STEM,87,47
2,Pre,Control,Yes,Man,Yes,L,Hispanic/Latino,Senior,Public Health,Non-STEM,71,42
3,Pre,Control,No,Woman,Yes,U,Asian,Junior,Biological Sciences,STEM,80,50
4,Pre,Intervention,No,Woman,Yes,U,Hispanic/Latino,Senior,Biological Sciences,STEM,86,49
5,Pre,Intervention,Yes,Man,Yes,L,Hispanic/Latino,Senior,Biological Sciences,STEM,86,55
6,Pre,Control,No,Woman,Yes,U,Asian,Senior,Biological Sciences,STEM,82,43


In [ ]:
head(post_survey_clean)

DID #,Survey,Condition,First Generation Status,Gender,Transfer Student,Upper v Lower,Race/Ethnicity,Class Level,Major,STEM,J_total,T_total
<dbl>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<chr>,<fct>,<dbl>,<dbl>
1,Post,Control,Yes,Man,No,L,Asian,Sophomore,Nursing,Non-STEM,96,48
2,Post,Control,Yes,Man,Yes,L,Hispanic/Latino,Senior,Public Health,Non-STEM,79,45
3,Post,Control,No,Woman,Yes,U,Asian,Junior,Biological Sciences,STEM,95,53
4,Post,Intervention,No,Woman,Yes,U,Hispanic/Latino,Senior,Biological Sciences,STEM,86,48
5,Post,Intervention,Yes,Man,Yes,L,Hispanic/Latino,Senior,Biological Sciences,STEM,98,56
6,Post,Control,No,Woman,Yes,U,Asian,Senior,Biological Sciences,STEM,86,53


In [ ]:
head(matched_change)

DID #,Condition,STEM,Major,Gender,Transfer Student,Upper v Lower,Race/Ethnicity,Class Level,First Generation Status,J_pre,J_post,J_change,T_pre,T_post,T_change
<dbl>,<fct>,<fct>,<chr>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,Control,Non-STEM,Nursing,Man,No,L,Asian,Sophomore,Yes,87,96,9,47,48,1
2,Control,Non-STEM,Public Health,Man,Yes,L,Hispanic/Latino,Senior,Yes,71,79,8,42,45,3
3,Control,STEM,Biological Sciences,Woman,Yes,U,Asian,Junior,No,80,95,15,50,53,3
4,Intervention,STEM,Biological Sciences,Woman,Yes,U,Hispanic/Latino,Senior,No,86,86,0,49,48,-1
5,Intervention,STEM,Biological Sciences,Man,Yes,L,Hispanic/Latino,Senior,Yes,86,98,12,55,56,1
6,Control,STEM,Biological Sciences,Woman,Yes,U,Asian,Senior,No,82,86,4,43,53,10
